# AUV-Swarm-FL-RL | Kaggle Execution Hub

Notebook này được thiết kế để chạy trên Kaggle Notebook và thực thi pipeline theo thứ tự tối ưu thời gian:
1) Beta sensitivity (Fig 1, 2, 3)
2) Train PPO-only bootstrap model
3) Scheme comparison / ablation (Fig 4, 5, 6)
4) Train + plot baselines medium (Fig 7 - 100 rounds)
5) Train + plot baselines full (Fig 7 - 1000 rounds)
6) Zip kết quả vào /kaggle/working để tải về

## 1. Clone Source Code và Cài Đặt Môi Trường (Kaggle)
Cập nhật URL repository bên dưới về repo GitHub của bạn trước khi chạy cell.

In [ ]:
import os
import shutil

# TODO: Cập nhật URL repo của bạn
REPO_URL = "https://github.com/ngnam1104/AUV-Swarm-RFL.git"
WORKDIR = "/kaggle/working"
REPO_DIR = os.path.join(WORKDIR, "AUV-Swarm-RFL")

# Xóa bản clone cũ nếu tồn tại
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Clone repo vào thư mục làm việc của Kaggle
!git clone $REPO_URL $REPO_DIR

# Di chuyển vào repo
os.chdir(REPO_DIR)

# Tạo thư mục log giống pipeline PowerShell
os.makedirs("results/logs", exist_ok=True)

# Cài đặt dependencies
!pip install -r requirements.txt

print("Kaggle environment is ready")
print("Current working directory:", os.getcwd())

In [ ]:
from torchvision import datasets

datasets.MNIST(root='./data', download=True)

## 2. Bước 1 - Beta Sensitivity (Figure 1, 2, 3)
Chạy thí nghiệm độ nhạy của hệ số beta tương tự bước đầu tiên trong script PowerShell.

In [ ]:
!python -u scripts/eval_beta_sensitivity.py \
  --rounds 50 \
  --m-values 9 16 25 \
  --out-dir results/beta_sensitivity

## 3. Bước 2 - Train PPO-Only Bootstrap Model
Chỉ huấn luyện PPO để tạo model đầu vào cho thí nghiệm Figure 4, 5, 6 (nhanh hơn train đủ 4 thuật toán).

In [ ]:
!python -u scripts/train_baselines.py \
  --episodes 50 \
  --m 9 \
  --max-fl-rounds 1000 \
  --algorithms ppo \
  --print-every-steps 1 \
  --out-dir results/ppo_for_fig456 \
  --step-log-file results/logs/ppo_for_fig456_step_log.txt

## 4. Bước 3 - Scheme Comparison / Ablation (Figure 4, 5, 6)
Chạy Figure 4, 5, 6 ngay sau khi đã có PPO bootstrap model.

In [ ]:
!python -u scripts/run_fig_4_5_6.py \
  --rounds 1000 \
  --m-values 9 16 25 36 49 \
  --out-dir results/eval_schemes \
  --model-path results/ppo_for_fig456/ppo_baseline_model

## 5. Bước 4 - Train Baselines Medium (Figure 7 - 100 rounds)
Huấn luyện đầy đủ 4 baseline cho Figure 7 (medium setting).

In [ ]:
!python -u scripts/train_baselines.py \
  --episodes 50 \
  --m 9 \
  --max-fl-rounds 100 \
  --algorithms ppo ddpg greedy random \
  --print-every-steps 1 \
  --out-dir results/fig_7_medium \
  --step-log-file results/logs/baselines_step_log_medium.txt

## 6. Bước 5 - Plot Figure 7 Medium
Vẽ biểu đồ Figure 7 cho kết quả medium (100 rounds).

In [ ]:
!python -u scripts/plot_fig_7.py \
  --input-dir results/fig_7_medium \
  --sigma 2.0 \
  --out-path results/fig_7_medium/figure7_medium.png

from IPython.display import Image, display
display(Image("results/fig_7_medium/figure7_medium.png"))

## 7. Bước 6 - Train Baselines Full (Figure 7 - 1000 rounds)
Huấn luyện đầy đủ 4 baseline cho Figure 7 (full setting).

In [ ]:
!python -u scripts/train_baselines.py \
  --episodes 50 \
  --m 9 \
  --max-fl-rounds 1000 \
  --algorithms ppo ddpg greedy random \
  --print-every-steps 1 \
  --out-dir results/fig_7_full \
  --step-log-file results/logs/baselines_step_log_full.txt

## 8. Bước 7 - Plot Figure 7 Full và Zip Kết Quả
Vẽ Figure 7 full, sau đó nén toàn bộ thư mục results vào /kaggle/working để tải về.

In [ ]:
!python -u scripts/plot_fig_7.py \
  --input-dir results/fig_7_full \
  --sigma 2.0 \
  --out-path results/fig_7_full/figure7_full.png

from IPython.display import Image, display
display(Image("results/fig_7_full/figure7_full.png"))

ZIP_PATH = "/kaggle/working/experiment_results.zip"
!zip -r $ZIP_PATH results/

print("Zip file saved at:", ZIP_PATH)
print("Open the right panel in Kaggle (Output/Files) to download experiment_results.zip")